# Train Actor-Critic

Actor-Critic updates a stochastic actor from the critic's one-step TD advantage:

$$\delta_t=r_{t+1}+\gamma V_\phi(s_{t+1})-V_\phi(s_t),\qquad \mathcal L_\pi=-\log\pi_\theta(a_t\mid s_t)\,\delta_t.$$

Here $\delta_t$ is the TD advantage, $V_\phi$ the critic, $\pi_\theta$ the actor, and $\gamma$ the discount factor. This notebook trains it on `CartPole-v1` with a categorical policy, fixed-length rollouts, and entropy regularization.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from aprenderl import ActorCritic, ActorCriticConfig
from aprenderl.utils import evaluate_policy

ENV_ID = "CartPole-v1"

In [ ]:
env = gym.make(ENV_ID)
config = ActorCriticConfig(
    learning_rate=1e-3,
    value_learning_rate=1e-3,
    gamma=0.99,
    n_steps=32,
    entropy_coefficient=1e-3,
)

agent = ActorCritic(env, config=config, device="cpu")
agent.learn(total_timesteps=20_000)
env.close()

In [ ]:
returns = np.asarray(agent.episode_returns)
window = min(20, len(returns))
moving_average = np.convolve(returns, np.ones(window) / window, mode="valid")

plt.figure(figsize=(8, 4))
plt.plot(returns, alpha=0.35, label="Episode return")
plt.plot(
    np.arange(window - 1, len(returns)),
    moving_average,
    label=f"{window}-episode average",
)
plt.xlabel("Episode")
plt.ylabel("Return")
plt.title(f"Actor-Critic training on {ENV_ID}")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## Watch the trained policy

This opens a window and runs 5 episodes using deterministic actions.

In [ ]:
evaluation_env = gym.make(ENV_ID, render_mode="human")
try:
    result = evaluate_policy(
        agent, evaluation_env, episodes=5, deterministic=True
    )
finally:
    evaluation_env.close()

print("Episode returns:", result.returns)
print(f"Mean return: {result.mean_return:.1f} +/- {result.return_std:.1f}")